# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [9]:
# TODO
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## A.2. Missing values & Duplicate data

In [10]:
# TODO
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(1)

## A.3. Invalid values

In [2]:
# TODO
invalid_age = (df['age'] < 0).sum()
invalid_bmi = (df['bmi'] <= 0).sum()
invalid_children = (df['children'] < 0).sum()
invalid_charges = (df['charges'] <= 0).sum()

print(invalid_age)
print(invalid_bmi)
print(invalid_children)
print(invalid_charges)



0
0
0
0


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [48]:
# TODO
df['bmi_group'] = pd.cut(
        df['bmi'],
        bins=[-np.inf, 25, 30, np.inf],
        labels=['Normal', 'Overweight', 'Obese']
)
df[['bmi','bmi_group']].head()

,bmi,bmi_group
0,27.900,Overweight
1,33.770,Obese
2,33.000,Obese
3,22.705,Normal
4,28.880,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [14]:
# TODO
nums_col = ['age', 'bmi', 'children', 'charges']

central_tendency = pd.DataFrame({
        'Mean': df[nums_col].mean(),
        'Median': df[nums_col].median(),
        'Mode': df[nums_col].mode().iloc[0] 
})
central_tendency = central_tendency.round(2)

In [17]:
df[nums_col].describe().round(2)

,age,bmi,children,charges
count,1338.00,1338.00,1338.00,1338.00
mean,39.21,30.66,1.09,13270.42
std,14.05,6.10,1.21,12110.01
min,18.00,15.96,0.00,1121.87
25%,27.00,26.30,0.00,4740.29
50%,39.00,30.40,1.00,9382.03
75%,51.00,34.69,2.00,16639.91
max,64.00,53.13,5.00,63770.43


## Group 2 — Dispersion

In [23]:
# TODO
nums_col = ['age', 'bmi', 'children', 'charges']

sub = df[nums_col]
q1 = sub.quantile(0.25)
q3 = sub.quantile(0.75)

dispersion_df = pd.DataFrame({
        'Range': sub.max() - sub.min(),
        'Std': sub.std(),
        'IQR': q3 - q1,
        'CV': sub.std() / sub.mean()
}) 
print(dispersion_df.round(2))

             Range       Std       IQR    CV
age          46.00     14.05     24.00  0.36
bmi          37.17      6.10      8.40  0.20
children      5.00      1.21      2.00  1.10
charges   62648.55  12110.01  11899.63  0.91


## Group 3 — Location and Shape

In [24]:
# TODO
shape_df = pd.DataFrame({
    'Skewness': df[nums_col].skew(),
    'Kurtosis': df[nums_col].kurtosis(),
    'Q1 (25%)': df[nums_col].quantile(0.25),
    'Q3 (75%)': df[nums_col].quantile(0.75),
    'Median': df[nums_col].median(),
    'P90 (90%)': df[nums_col].quantile(0.9),
})
print(shape_df.round(2))

          Skewness  Kurtosis  Q1 (25%)  Q3 (75%)   Median  P90 (90%)
age           0.06     -1.25     27.00     51.00    39.00      59.00
bmi           0.28     -0.05     26.30     34.69    30.40      38.62
children      0.94      0.20      0.00      2.00     1.00       3.00
charges       1.52      1.61   4740.29  16639.91  9382.03   34831.72


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [31]:
# TODO
ovr_charges = df.groupby('smoker')['charges'].mean()
ovr_ratio = ovr_charges['yes'] / ovr_charges['no']

print(ovr_charges.round(2))
print(f"Số người hút thuốc nhiều hơn số người không hút là: {ovr_ratio.round(2)}")

smoker
no      8434.27
yes    32050.23
Name: charges, dtype: float64
Số người hút thuốc nhiều hơn số người không hút là: 3.8


In [32]:
region_comp = df.groupby(['region', 'smoker'])['charges'].mean().unstack()
region_comp['Ratio'] = region_comp['yes'] / region_comp['no']
print(region_comp.round(2))

smoker          no       yes  Ratio
region                             
northeast  9165.53  29673.54   3.24
northwest  8556.46  30192.00   3.53
southeast  8032.22  34845.00   4.34
southwest  8019.28  32269.06   4.02


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [ ]:
# TODO
corr_smoker = df[df['smoker'] == 'yes']['bmi'].corr(df[df['smoker'] == 'yes']['charges'])
corr_non_smoker = df[df['smoker'] == 'no']['bmi'].corr(df[df['smoker'] == 'no']['charges'])
print(corr_smoker.round(2)) 
print(corr_non_smoker.round(2))

0.81
0.08


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [41]:
# TODO
region_charges = df.groupby('region')['charges'].agg(['mean', 'median', 'count']).sort_values(by='mean', ascending=False)
print(region_charges.round(2))

               mean    median  count
region                              
southeast  14735.41   9294.13    364
northeast  13406.38  10057.65    324
northwest  12417.58   8965.80    325
southwest  12346.94   8798.59    325


In [43]:
highest_region = region_charges.index[0]
highest_val = region_charges['mean'].iloc[0]
print(highest_region)
print(highest_val.round(2))

southeast
14735.41


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [44]:
# TODO
children_stats = df.groupby('children')['charges'].agg(['mean', 'median', 'count']).sort_values(by='mean', ascending=False)
print(children_stats.round(2))

              mean    median  count
children                           
3         15355.32  10600.55    157
2         15073.56   9264.98    240
4         13850.66  11033.66     25
1         12731.17   8483.87    324
0         12365.98   9856.95    574
5          8786.04   8589.57     18


In [46]:
corr_children = df['children'].corr(df['charges'])
print(corr_children.round(2))

0.07


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [47]:
# TODO
corr_age = df['age'].corr(df['charges'])
print(corr_age.round(2))

0.3


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*